# Deploy whisper-base to aws SageMaker

In [7]:
import numpy as np
import torch
import whisper
import sagemaker
import time
import os

In [ ]:
model = whisper.load_model("base")
print(
    f"Model is {'multilingual' if model.is_multilingual else 'English-only'} "
    f"and has {sum(np.prod(p.shape) for p in model.parameters()):,} parameters."
)

100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 298MiB/s]


Model is multilingual and has 71,825,920 parameters.


In [5]:
sess = sagemaker.session.Session()
bucket = sess.default_bucket()
prefix = 'whisper-demo-deploy/'
s3_uri = f's3://{bucket}/{prefix}'

In [25]:
torch.save(
    {
        'model_state_dict': model.state_dict(),
        'dims': model.dims.__dict__,
    },
    'base.pt'
)

Once the model has been saved, you will package the model into a tar.gz file and upload it to Amazon S3. This serialized model will be the model artifact which is referenced for real-time inference.

In [26]:
!mkdir -p model
!mv base.pt model
!cd model && tar -czvf model.tar.gz base.pt
!mv model/model.tar.gz .
!tar -tvf model.tar.gz
model_uri = sess.upload_data('model.tar.gz', bucket = bucket, key_prefix=f"{prefix}model")
!rm model.tar.gz
!rm -rf model

base.pt
-rw-r--r-- sagemaker-user/users 290444708 2025-11-28 11:05 base.pt


In [ ]:
# model_uri = '' # Set model URI here if already uploaded
model_uri


## Create SageMaker Model Object

Once the model artifact has been uploaded to S3, you will use the SageMaker SDK to create a `model` object which references the model artifact in S3, one of SageMaker's PyTorch inference containers, and the inference code stored in the `src` directory in this repository. The `inference.py` is the code which is executed at runtime while the `requirements.txt` tells SageMaker to install the `whisper` library inside its Docker container.

In [ ]:
image = sagemaker.image_uris.retrieve(
    framework='pytorch',
    region='eu-central-1',
    image_scope='inference',
    version='1.12',
    instance_type='ml.g4dn.xlarge',
)

model_name = f'whisper-model-{int(time.time())}'
whisper_model_sm = sagemaker.model.Model(
    model_data=model_uri,
    image_uri=image,
    role='', # Replace with SageMaker Execution Role
    entry_point="inference.py",
    source_dir='src',
    name=model_name,
)

## Deploy to a Real Time Endpoint

Deploying the `model` object to sagemaker can be done with the `deploy` function. Notice that you will be using a `ml.g4dn.xlarge` instance type in order to take advantage of a AWS's low cost GPU instances for accelerated inference.

In [ ]:
endpoint_name = f'whisper-endpoint-cpu'
whisper_model_sm.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.2xlarge",
    endpoint_name=endpoint_name,
    wait=True,
)

Using already existing model: whisper-model-1764337619


----------!

In [ ]:
endpoint_name = f'whisper-endpoint-gpu'
whisper_model_sm.deploy(
    initial_instance_count=1,
    instance_type="ml.g4dn.xlarge",
    endpoint_name=endpoint_name,
    update_endpoint=True,  # Add this to update existing endpoint
    wait=True,
)


In [29]:
endpoint_name

'whisper-endpoint-gpu'

## Test Inference

You can test the endpoint with the test.py script
